# Preparación y Análisis Exploratorio de Datos
## Dataset: Walmart Sales
### Materia: Inteligencia Artificial

---

**Descripción del Dataset:**  
El dataset contiene registros históricos de ventas semanales de tiendas Walmart, junto con variables contextuales como temperatura, precio del combustible, CPI (Índice de Precios al Consumidor), tasa de desempleo y si la semana corresponde a un día festivo.

**Variable objetivo:** `Weekly_Sales`

---
## 0. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías importadas correctamente.')

: 

---
## 1. Carga de Datos

### 1.1 Función de carga

In [ ]:
def cargar_datos(ruta: str) -> pd.DataFrame:
    """
    Carga un archivo CSV desde la ruta especificada y
    convierte la columna 'Date' al tipo datetime.

    Parámetros:
        ruta (str): Ruta del archivo CSV.

    Retorna:
        pd.DataFrame: DataFrame con los datos cargados.
    """
    try:
        df = pd.read_csv(ruta, parse_dates=['Date'], dayfirst=True)
        print(f'✅ Archivo cargado exitosamente.')
        print(f'   → Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
        return df
    except FileNotFoundError:
        print(f'❌ Error: No se encontró el archivo en "{ruta}".')
        return None


# ── Carga del dataset ──────────────────────────────────────────────────────────
RUTA_CSV = 'Walmart_Sales.csv'   # Ajusta la ruta si es necesario
df = cargar_datos(RUTA_CSV)

### 1.2 Primeros y últimos registros

In [ ]:
print('═' * 60)
print('PRIMEROS 5 REGISTROS')
print('═' * 60)
display(df.head())

print('\n')
print('═' * 60)
print('ÚLTIMOS 5 REGISTROS')
print('═' * 60)
display(df.tail())

### 1.3 Estructura del DataFrame

In [ ]:
print('═' * 60)
print('INFORMACIÓN GENERAL DEL DATAFRAME')
print('═' * 60)
df.info()

print('\n')
print('═' * 60)
print('TIPOS DE DATOS POR COLUMNA')
print('═' * 60)
display(df.dtypes.rename('Tipo').to_frame())

---
## 2. Limpieza de Datos

### 2.1 Conteo de valores nulos y duplicados

In [ ]:
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Valores Nulos': nulos,
    '% del Total': pct_nulos
})

print('═' * 60)
print('VALORES NULOS POR COLUMNA')
print('═' * 60)
display(resumen_nulos)

print(f'\n═' * 60)
duplicados = df.duplicated().sum()
print(f'FILAS DUPLICADAS: {duplicados}')

### 2.2 Eliminación de duplicados

In [ ]:
filas_antes = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
filas_despues = len(df)

print(f'Filas antes de eliminar duplicados : {filas_antes}')
print(f'Filas después de eliminar duplicados: {filas_despues}')
print(f'Registros eliminados               : {filas_antes - filas_despues}')

### 2.3 Completar valores faltantes en `Weekly_Sales` (con la mediana)

In [ ]:
mediana_sales = df['Weekly_Sales'].median()

print(f'Mediana de Weekly_Sales: {mediana_sales:,.2f}')
print(f'Nulos antes del relleno: {df["Weekly_Sales"].isnull().sum()}')

df['Weekly_Sales'].fillna(mediana_sales, inplace=True)

print(f'Nulos después del relleno: {df["Weekly_Sales"].isnull().sum()}')
print('✅ Valores faltantes de Weekly_Sales completados con la mediana.')

### 2.4 Eliminar el resto de filas con valores faltantes

In [ ]:
filas_antes = len(df)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
filas_despues = len(df)

print(f'Filas antes : {filas_antes}')
print(f'Filas después: {filas_despues}')
print(f'Filas eliminadas: {filas_antes - filas_despues}')
print(f'\nNulos restantes por columna:')
print(df.isnull().sum())

### 2.5 Estadística general del dataset

In [ ]:
print('═' * 60)
print('ESTADÍSTICA DESCRIPTIVA')
print('═' * 60)
display(df.describe().T)

### 2.6 Histogramas y gráficas de cajas

In [ ]:
variables_num = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

fig, axes = plt.subplots(nrows=len(variables_num), ncols=2,
                          figsize=(14, 4 * len(variables_num)))
fig.suptitle('Histogramas y Diagramas de Caja por Variable', fontsize=16, fontweight='bold', y=1.01)

for i, col in enumerate(variables_num):
    # Histograma
    axes[i, 0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i, 0].set_title(f'Histograma – {col}', fontsize=11)
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frecuencia')

    # Boxplot
    axes[i, 1].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightcoral', color='firebrick'),
                       medianprops=dict(color='black', linewidth=2),
                       whiskerprops=dict(color='firebrick'),
                       capprops=dict(color='firebrick'),
                       flierprops=dict(marker='o', color='firebrick', alpha=0.4, markersize=4))
    axes[i, 1].set_title(f'Diagrama de Caja – {col}', fontsize=11)
    axes[i, 1].set_ylabel(col)

plt.tight_layout()
plt.savefig('histogramas_boxplots.png', bbox_inches='tight')
plt.show()
print('✅ Gráficas generadas y guardadas.')

### 2.7 Corrección / Eliminación de valores atípicos (outliers)

Se usa el método **IQR (Rango Intercuartílico)** para detectar outliers en las variables numéricas.  
Los registros atípicos se guardan en un CSV separado llamado `anomalias.csv`.

In [ ]:
def detectar_outliers_iqr(dataframe: pd.DataFrame, columnas: list) -> pd.Series:
    """
    Identifica filas con valores atípicos en al menos una de las columnas
    indicadas, usando el criterio IQR (Q1 - 1.5*IQR  /  Q3 + 1.5*IQR).

    Retorna:
        pd.Series (bool): True donde la fila es atípica.
    """
    mascara = pd.Series(False, index=dataframe.index)
    for col in columnas:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        limite_inf = Q1 - 1.5 * IQR
        limite_sup = Q3 + 1.5 * IQR
        outliers_col = (dataframe[col] < limite_inf) | (dataframe[col] > limite_sup)
        print(f'  {col:>15}: {outliers_col.sum():>5} outliers  '
              f'(límites: [{limite_inf:.2f}, {limite_sup:.2f}])')
        mascara = mascara | outliers_col
    return mascara


print('Detección de outliers por columna (método IQR):')
print('─' * 60)
mascara_outliers = detectar_outliers_iqr(df, variables_num)

# Separar outliers
df_anomalias = df[mascara_outliers].copy()
df_limpio    = df[~mascara_outliers].copy()
df_limpio.reset_index(drop=True, inplace=True)

print(f'\nTotal de registros atípicos  : {len(df_anomalias)}')
print(f'Total de registros limpios   : {len(df_limpio)}')

# Guardar anomalías
df_anomalias.to_csv('anomalias.csv', index=False)
print('\n✅ Outliers guardados en "anomalias.csv".')
display(df_anomalias.head())

In [ ]:
# Actualizar df con los datos limpios
df = df_limpio.copy()
print(f'DataFrame de trabajo actualizado → {df.shape[0]} filas × {df.shape[1]} columnas')

---
## 3. Selección de Atributos (Feature Selection) *(Opcional)*

### 3.1 Dispersión de cada variable respecto a la variable objetivo (`Weekly_Sales`)

In [ ]:
variables_independientes = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Holiday_Flag', 'Store']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
axes = axes.flatten()
fig.suptitle('Dispersión de Variables vs. Weekly_Sales', fontsize=15, fontweight='bold')

for i, col in enumerate(variables_independientes):
    axes[i].scatter(df[col], df['Weekly_Sales'],
                    alpha=0.3, s=10, color='steelblue')
    # Línea de tendencia
    z = np.polyfit(df[col], df['Weekly_Sales'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 200)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Tendencia')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Weekly_Sales')
    axes[i].set_title(f'{col} vs Weekly_Sales')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('dispersion_variables.png', bbox_inches='tight')
plt.show()
print('✅ Gráficas de dispersión generadas.')

### 3.2 Mapa de correlación

In [ ]:
cols_correlacion = variables_independientes + ['Weekly_Sales']
corr_matrix = df[cols_correlacion].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            vmin=-1, vmax=1, center=0)
ax.set_title('Mapa de Correlación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mapa_correlacion.png', bbox_inches='tight')
plt.show()

print('\nCorrelaciones con Weekly_Sales (ordenadas):')
display(corr_matrix['Weekly_Sales'].drop('Weekly_Sales').sort_values(key=abs, ascending=False))

### 3.3 Atributos sin relación con la variable objetivo

Se consideran **sin relación** aquellas variables cuya correlación con `Weekly_Sales` es menor a **0.05** en valor absoluto.  
Se guardan en un CSV separado llamado `atributosSinRelacion.csv`.

In [ ]:
UMBRAL_CORRELACION = 0.05

correlaciones = corr_matrix['Weekly_Sales'].drop('Weekly_Sales')
sin_relacion  = correlaciones[abs(correlaciones) < UMBRAL_CORRELACION].index.tolist()
con_relacion  = correlaciones[abs(correlaciones) >= UMBRAL_CORRELACION].index.tolist()

print(f'Umbral de correlación   : |r| < {UMBRAL_CORRELACION}')
print(f'Atributos SIN relación  : {sin_relacion}')
print(f'Atributos CON relación  : {con_relacion}')

if sin_relacion:
    df_sin_relacion = df[sin_relacion].copy()
    df_sin_relacion.to_csv('atributosSinRelacion.csv', index=False)
    print('\n✅ Atributos sin relación guardados en "atributosSinRelacion.csv".')
    display(df_sin_relacion.head())
else:
    print('\nℹ️  Ningún atributo cae por debajo del umbral → no se generó el archivo.')

### 3.4 DataFrame final con atributos seleccionados

In [ ]:
# Columnas a conservar: atributos CON relación + variable objetivo
cols_finales = ['Date'] + con_relacion + ['Weekly_Sales']
df_final = df[cols_finales].copy()

print('═' * 60)
print('DATAFRAME FINAL LISTO PARA MODELADO')
print('═' * 60)
print(f'Dimensiones: {df_final.shape[0]} filas × {df_final.shape[1]} columnas')
print(f'Columnas   : {list(df_final.columns)}')
display(df_final.head())

df_final.to_csv('Walmart_Sales_limpio.csv', index=False)
print('\n✅ Dataset limpio guardado en "Walmart_Sales_limpio.csv".')

---
## Resumen del Proceso

| Etapa | Acción | Resultado |
|---|---|---|
| Carga | Lectura del CSV con parseo de fechas | 6 437 registros, 8 columnas |
| Duplicados | `drop_duplicates()` | Registros duplicados eliminados |
| Nulos – `Weekly_Sales` | Relleno con **mediana** | Sin pérdida de registros |
| Nulos – resto | `dropna()` | Filas incompletas eliminadas |
| Outliers | Criterio **IQR** | Separados en `anomalias.csv` |
| Feature Selection | Correlación \|r\| < 0.05 | Descartados en `atributosSinRelacion.csv` |
| Dataset final | Atributos relevantes | `Walmart_Sales_limpio.csv` |